# AssetOpsBench environment setup

A beginner-friendly setup guide for the KDD tutorial. Complete this notebook **before** `00_mcp_agents_architecture.ipynb`. The commands come from the repository's [INSTRUCTIONS.md](../../INSTRUCTIONS.md), with additional explanations and checks for first-time users.

> Some setup must be performed in a terminal before a notebook kernel can run. Those commands are shown as terminal blocks; do not paste them into Python cells.

## What you will set up

1. Git and the AssetOpsBench repository
2. Python 3.12+ and the `uv` environment manager
3. Project dependencies in `.venv`
4. A local `.env` configuration file
5. Docker and CouchDB with the tutorial's default data
6. The correct VS Code/Jupyter kernel
7. A final readiness check


## Before installing: what runs where?

| Component | Where it runs | Why it is needed |
|---|---|---|
| VS Code or JupyterLab | Your computer | Opens and executes notebooks |
| `.venv` Python environment | Repository directory | Contains the exact Python packages and MCP command entry points |
| CouchDB | Docker container | Stores assets, sensors, work orders, failure codes, catalogs, and other tutorial records |
| MCP servers | On-demand local subprocesses | Expose repository capabilities through MCP; notebook/agent code launches them when needed |
| Model provider | Remote service | Required only for model-backed generation, Stirrup execution, or LLM judging |

You do **not** need to start every MCP server in a separate terminal. The notebooks and agent runners launch the stdio servers on demand.


## 1. Install the prerequisites

Install these before continuing:

- **Git** — downloads and updates the repository.
- **Python 3.12 or newer** — required by `pyproject.toml`.
- **uv** — creates the virtual environment and installs dependencies.
- **Docker Desktop** (or an equivalent Docker engine) — runs CouchDB.
- **VS Code** with the Python and Jupyter extensions, or JupyterLab.

Check what is already installed in a terminal:

```bash
git --version
python3 --version
uv --version
docker --version
docker compose version
```

Install `uv` on macOS/Linux with either command:

```bash
brew install uv
# or
curl -LsSf https://astral.sh/uv/install.sh | sh
```

> Windows participants should preferably use WSL2 with Docker Desktop integration. Run all repository commands from the same environment; do not mix Windows Python with a WSL repository checkout.


## 2. Get the repository and enter its root

For a new checkout:

```bash
git clone https://github.com/IBM/AssetOpsBench.git
cd AssetOpsBench
```

If you already have the repository, open a terminal in it and confirm the location:

```bash
pwd
ls
```

The root should contain `pyproject.toml`, `src`, `notebook`, and `INSTRUCTIONS.md`. Run the remaining terminal commands from this root—not from `notebook/kdd_tutorial`.


## 3. Create the Python environment

From the repository root:

```bash
uv sync
```

This creates `.venv`, installs the pinned project dependencies, and registers commands such as `iot-mcp-server`, `stirrup-agent`, and `plan-execute`. It may take several minutes the first time.

You can run commands without activating the environment by prefixing them with `uv run`. Alternatively, activate it for the current terminal session:

```bash
source .venv/bin/activate
```

Verify the environment:

```bash
uv run python --version
uv run python -c "import mcp; print('MCP SDK ready')"
```


## 4. Create the environment configuration

Create your private `.env` from the public template:

```bash
cp .env.public .env
```

Open `.env` in an editor. The default CouchDB values work with the bundled Docker configuration:

```dotenv
COUCHDB_URL=http://localhost:5984
COUCHDB_USERNAME=admin
COUCHDB_PASSWORD=password
```

Model credentials depend on the route used during the tutorial:

- WatsonX routes: `WATSONX_APIKEY`, `WATSONX_PROJECT_ID`, and optionally `WATSONX_URL`.
- LiteLLM proxy routes: `LITELLM_API_KEY` and `LITELLM_BASE_URL`.
- TokenRouter routes: `TOKENROUTER_API_KEY` and `TOKENROUTER_BASE_URL`.

Utilities and many deterministic examples do not require model credentials. Stirrup and LLM-as-judge evaluation do.

> Never place real credentials directly in a notebook, commit `.env`, print secret values, or share them in screenshots.


## 5. Start Docker and CouchDB

First launch Docker Desktop and wait until its engine reports that it is running. Check in a terminal:

```bash
docker info
```

Then start the repository's CouchDB service from the repository root:

```bash
docker compose -f src/couchdb/docker-compose.yaml up -d
```

The container starts CouchDB on port `5984` and loads the **default tutorial data automatically**. Wait until it becomes healthy:

```bash
docker compose -f src/couchdb/docker-compose.yaml ps
curl -s http://localhost:5984/
curl -s -u admin:password http://localhost:5984/_all_dbs
```

You can also open the CouchDB Fauxton interface at [http://localhost:5984/_utils](http://localhost:5984/_utils) and sign in with `admin` / `password`.

To stop CouchDB without deleting its Docker volume:

```bash
docker compose -f src/couchdb/docker-compose.yaml stop
```

> Avoid `docker compose down -v` during the tutorial: `-v` removes the CouchDB volume.


## 6. Understand the tutorial data

The Compose startup loads the default manifest from `src/couchdb/scenarios_data/default/manifest.json`. Shared input files live under `src/couchdb/scenarios_data/shared/`. Each manifest key becomes a CouchDB database with the same name.

Normally, no manual loading is needed. If an instructor asks you to restore the default data without restarting Docker, run:

```bash
uv run python src/couchdb/init_data.py
```

Scenario-specific loading is available, but it changes the database contents:

```bash
uv run python src/couchdb/init_data.py SCENARIO_ID --reset
```

> Use scenario loading only when the tutorial explicitly requests it. `--reset` drops user databases before loading the selected scenario.


## 7. Open the tutorial with the correct kernel

### VS Code

1. Open the **AssetOpsBench repository root** as the VS Code folder.
2. Install the Microsoft Python and Jupyter extensions if prompted.
3. Open `notebook/kdd_tutorial/00_environment_setup.ipynb`.
4. Click **Select Kernel** in the upper-right corner.
5. Choose the interpreter at `AssetOpsBench/.venv/bin/python`.

### JupyterLab

Start it from the repository root so relative paths resolve consistently:

```bash
uv run jupyter lab
```

> A frequent error is using the system Python kernel instead of `.venv`. If `import mcp` fails, reselect the kernel before reinstalling anything.


## 8. Run the read-only readiness check

The next cell does not install packages, start containers, load data, or display secret values. It only reports whether the expected pieces are available.


In [1]:
from pathlib import Path
import importlib.util
import os
import shutil
import socket
import subprocess
import sys

def find_repo(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    return None

def command_version(command):
    path = shutil.which(command)
    if not path:
        return "not found"
    try:
        result = subprocess.run(
            [command, "--version"], capture_output=True, text=True, timeout=10
        )
        return (result.stdout or result.stderr).strip().splitlines()[0]
    except Exception as exc:
        return f"found at {path}, but version check failed: {exc}"

def tcp_reachable(host, port, timeout=1.5):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

repo = find_repo()
status = {
    "repository root": str(repo) if repo else "not found",
    "Python": sys.version.split()[0],
    "Python >= 3.12": sys.version_info >= (3, 12),
    "running from .venv": Path(sys.prefix).name == ".venv",
    "uv": command_version("uv"),
    "docker": command_version("docker"),
    "MCP Python package": importlib.util.find_spec("mcp") is not None,
    ".env exists": bool(repo and (repo / ".env").exists()),
    "CouchDB port 5984 reachable": tcp_reachable("localhost", 5984),
}

for name, value in status.items():
    print(f"{name:32} {value}")

if not all([status["Python >= 3.12"], status["running from .venv"], status["MCP Python package"]]):
    print("\nPython environment needs attention. Run `uv sync` and select .venv/bin/python.")
if not status["CouchDB port 5984 reachable"]:
    print("\nCouchDB is not reachable. Start Docker Desktop, then run the Compose command above.")


repository root                  /Users/chathurangishyalika/IBM/AssetOpsBench
Python                           3.12.13
Python >= 3.12                   True
running from .venv               False
uv                               uv 0.11.19 (Homebrew 2026-06-03 aarch64-apple-darwin)
docker                           Docker version 29.1.4-rd, build 3c6914c
MCP Python package               True
.env exists                      True
CouchDB port 5984 reachable      True

Python environment needs attention. Run `uv sync` and select .venv/bin/python.


## 9. Check credential names without exposing values

This optional cell reports only whether variables are configured. It never prints their contents. Restart the kernel after changing `.env`.


In [2]:
from dotenv import load_dotenv

if repo:
    load_dotenv(repo / ".env", override=False)

credential_groups = {
    "WatsonX": ["WATSONX_APIKEY", "WATSONX_PROJECT_ID"],
    "LiteLLM proxy": ["LITELLM_API_KEY", "LITELLM_BASE_URL"],
    "TokenRouter": ["TOKENROUTER_API_KEY", "TOKENROUTER_BASE_URL"],
}

for group, names in credential_groups.items():
    configured = [name for name in names if os.getenv(name)]
    print(f"{group:18} {len(configured)}/{len(names)} required variables configured")


WatsonX            2/2 required variables configured
LiteLLM proxy      2/2 required variables configured
TokenRouter        2/2 required variables configured


## 10. Troubleshooting

| Symptom | Likely cause | What to do |
|---|---|---|
| `uv: command not found` | `uv` is absent or not on `PATH` | Install it, restart the terminal, and run `uv --version` |
| `import mcp` fails | Wrong kernel or incomplete `uv sync` | Select `.venv/bin/python`; rerun `uv sync` from the root |
| `Cannot connect to the Docker daemon` | Docker Desktop/engine is stopped | Start Docker and wait; then run `docker info` |
| Port `5984` is unavailable | Another CouchDB/service owns the port | Run `docker ps`; stop the conflicting service or change configuration |
| CouchDB is reachable but data is missing | Initialization is incomplete or a scenario changed the data | Check Compose logs; restore the default data with `uv run python src/couchdb/init_data.py` |
| `Unknown tool` | Notebook uses a stale tool name | Run `list_tools()` and use the live name and schema |
| Agent times out | Model/tool-calling route, credentials, or excessive turns | Verify the model route, use a known tool-calling model, simplify the prompt, and inspect saved logs |
| Notebook images do not render | Cached tab or incorrect relative path | Reopen the notebook and ensure its `assets` folder travels with it |

Useful diagnostics:

```bash
docker compose -f src/couchdb/docker-compose.yaml ps
docker compose -f src/couchdb/docker-compose.yaml logs --tail=100 couchdb
curl -s -u admin:password http://localhost:5984/_all_dbs
uv run python -c "import sys, mcp; print(sys.executable); print('MCP ready')"
```


## Final checklist

Before continuing, confirm:

- [ ] The repository is open at its root.
- [ ] Python is 3.12 or newer.
- [ ] `uv sync` completed successfully.
- [ ] The notebook kernel is `.venv/bin/python`.
- [ ] `.env` exists and secrets are not stored in notebooks.
- [ ] Docker is running.
- [ ] CouchDB responds on port 5984.
- [ ] The default CouchDB databases are present.
- [ ] At least one model-provider route is configured before the Stirrup notebook.

## Next

Continue to [`00_mcp_agents_architecture.ipynb`](00_mcp_agents_architecture.ipynb) to learn MCP concepts, AssetOpsBench architecture, direct tool execution, and the agent loop.
